In [1]:
%pip install -q qiskit qiskit-aer pylatexenc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator

print("Qiskit version:", qiskit.__version__)
print("Setup complete!")

Qiskit version: 2.6.0
Setup complete!


In [3]:
num_counting = 3
phi = 3 / 8

print("Counting qubits:", num_counting)
print("Target phase:", phi)
print("Expected phase bits:", "011")
print("Phase angle:", 2 * np.pi * phi)

Counting qubits: 3
Target phase: 0.375
Expected phase bits: 011
Phase angle: 2.356194490192345


In [4]:
target = num_counting

# 3 counting qubits + 1 target qubit
qc = QuantumCircuit(num_counting + 1, num_counting)

# Prepare target qubit in |1>
qc.x(target)

# Create superposition in counting register
for q in range(num_counting):
    qc.h(q)

# Apply controlled powers of the phase operation
for q in range(num_counting):
    angle = 2 * np.pi * phi * (2 ** q)
    qc.cp(angle, q, target)

# Apply inverse QFT to counting qubits
qc.append(
    QFTGate(num_counting).inverse(),
    list(range(num_counting))
)

# Measure counting qubits
qc.measure(range(num_counting), range(num_counting))

print(qc.draw())

     ┌───┐                            ┌─────────┐┌─┐      
q_0: ┤ H ├─■──────────────────────────┤0        ├┤M├──────
     ├───┤ │                          │         │└╥┘┌─┐   
q_1: ┤ H ├─┼─────────■────────────────┤1 qft_dg ├─╫─┤M├───
     ├───┤ │         │                │         │ ║ └╥┘┌─┐
q_2: ┤ H ├─┼─────────┼─────────■──────┤2        ├─╫──╫─┤M├
     ├───┤ │P(3π/4)  │P(3π/2)  │P(3π) └─────────┘ ║  ║ └╥┘
q_3: ┤ X ├─■─────────■─────────■──────────────────╫──╫──╫─
     └───┘                                        ║  ║  ║ 
c: 3/═════════════════════════════════════════════╩══╩══╩═
                                                  0  1  2 


In [5]:
simulator = AerSimulator()
compiled_qc = transpile(qc, simulator)

result = simulator.run(
    compiled_qc,
    shots=1024
).result()

counts = result.get_counts()
print("Measurement counts:", counts)

Measurement counts: {'011': 1024}


In [6]:
expected = "011"

assert counts == {expected: 1024}, (
    f"Unexpected measurement counts: {counts}"
)

estimated_phi = int(expected, 2) / (2 ** num_counting)

print("Theoretical phase:", phi)
print("Estimated phase:", estimated_phi)
print("Absolute error:", abs(phi - estimated_phi))
print("All validation checks passed!")

Theoretical phase: 0.375
Estimated phase: 0.375
Absolute error: 0.0
All validation checks passed!


In [7]:
phi = 0.30
target = num_counting

qc2 = QuantumCircuit(num_counting + 1, num_counting)

# Prepare target eigenstate |1>
qc2.x(target)

# Counting-register superposition
for q in range(num_counting):
    qc2.h(q)

# Controlled powers of U
for q in range(num_counting):
    angle = 2 * np.pi * phi * (2 ** q)
    qc2.cp(angle, q, target)

# Inverse QFT
qc2.append(
    QFTGate(num_counting).inverse(),
    list(range(num_counting))
)

qc2.measure(range(num_counting), range(num_counting))

# Simulate
compiled_qc2 = transpile(qc2, simulator)
result2 = simulator.run(compiled_qc2, shots=4096).result()
counts2 = result2.get_counts()

print("Measurement counts:", counts2)

Measurement counts: {'100': 183, '101': 88, '001': 224, '110': 45, '111': 58, '011': 1044, '000': 98, '010': 2356}


In [8]:
total_shots = sum(counts2.values())

estimates = sorted(
    [
        (int(bits, 2) / (2 ** num_counting), shots)
        for bits, shots in counts2.items()
    ],
    key=lambda item: item[1],
    reverse=True
)

print("Theoretical phase:", phi)
print("Most frequent estimates:")

for estimate, shots in estimates[:5]:
    print(
        f"Phase estimate: {estimate:.3f} | "
        f"Frequency: {shots / total_shots:.1%}"
    )

Theoretical phase: 0.3
Most frequent estimates:
Phase estimate: 0.250 | Frequency: 57.5%
Phase estimate: 0.375 | Frequency: 25.5%
Phase estimate: 0.125 | Frequency: 5.5%
Phase estimate: 0.500 | Frequency: 4.5%
Phase estimate: 0.000 | Frequency: 2.4%


In [9]:
total_shots = sum(counts2.values())

print("Target phase:", phi)
print("Total shots:", total_shots)
print("\nPhase estimates, ranked by frequency:")

for bits, shots in sorted(
    counts2.items(),
    key=lambda item: item[1],
    reverse=True
):
    estimate = int(bits, 2) / (2 ** num_counting)
    probability = shots / total_shots

    print(
        f"{bits} -> phase {estimate:.3f} | "
        f"{shots} shots | {probability:.1%}"
    )

Target phase: 0.3
Total shots: 4096

Phase estimates, ranked by frequency:
010 -> phase 0.250 | 2356 shots | 57.5%
011 -> phase 0.375 | 1044 shots | 25.5%
001 -> phase 0.125 | 224 shots | 5.5%
100 -> phase 0.500 | 183 shots | 4.5%
000 -> phase 0.000 | 98 shots | 2.4%
101 -> phase 0.625 | 88 shots | 2.1%
111 -> phase 0.875 | 58 shots | 1.4%
110 -> phase 0.750 | 45 shots | 1.1%


In [10]:
# Day 8: Four-counting-qubit QPE
num_counting_4 = 4
phi = 0.30
target_4 = num_counting_4

qc4 = QuantumCircuit(
    num_counting_4 + 1,
    num_counting_4
)

# Prepare the target eigenstate |1>
qc4.x(target_4)

# Create superposition in the counting register
for q in range(num_counting_4):
    qc4.h(q)

# Encode the phase using controlled powers of U
for q in range(num_counting_4):
    angle = 2 * np.pi * phi * (2 ** q)
    qc4.cp(angle, q, target_4)

# Extract the phase using the inverse QFT
qc4.append(
    QFTGate(num_counting_4).inverse(),
    list(range(num_counting_4))
)

# Measure the counting register
qc4.measure(
    range(num_counting_4),
    range(num_counting_4)
)

print(qc4.draw())

     ┌───┐                                          ┌─────────┐┌─┐         
q_0: ┤ H ├─■────────────────────────────────────────┤0        ├┤M├─────────
     ├───┤ │                                        │         │└╥┘┌─┐      
q_1: ┤ H ├─┼─────────■──────────────────────────────┤1        ├─╫─┤M├──────
     ├───┤ │         │                              │  qft_dg │ ║ └╥┘┌─┐   
q_2: ┤ H ├─┼─────────┼─────────■────────────────────┤2        ├─╫──╫─┤M├───
     ├───┤ │         │         │                    │         │ ║  ║ └╥┘┌─┐
q_3: ┤ H ├─┼─────────┼─────────┼──────────■─────────┤3        ├─╫──╫──╫─┤M├
     ├───┤ │P(3π/5)  │P(6π/5)  │P(12π/5)  │P(24π/5) └─────────┘ ║  ║  ║ └╥┘
q_4: ┤ X ├─■─────────■─────────■──────────■─────────────────────╫──╫──╫──╫─
     └───┘                                                      ║  ║  ║  ║ 
c: 4/═══════════════════════════════════════════════════════════╩══╩══╩══╩═
                                                                0  1  2  3 


In [11]:
compiled_qc4 = transpile(qc4, simulator)

result4 = simulator.run(
    compiled_qc4,
    shots=4096
).result()

counts4 = result4.get_counts()

print("Measurement counts:", counts4)

Measurement counts: {'1101': 1, '1110': 6, '1111': 7, '0100': 239, '0110': 121, '1000': 16, '0101': 3557, '1001': 10, '1100': 3, '1011': 7, '0011': 41, '0111': 36, '0001': 9, '0010': 24, '0000': 11, '1010': 8}


In [12]:
total_shots4 = sum(counts4.values())

print("Target phase:", phi)
print("Four-qubit phase resolution:", 1 / 16)
print("\nResults ranked by frequency:")

for bits, shots in sorted(
    counts4.items(),
    key=lambda item: item[1],
    reverse=True
):
    estimate = int(bits, 2) / 16

    print(
        f"{bits} -> phase {estimate:.4f} | "
        f"{shots} shots | "
        f"{shots / total_shots4:.1%}"
    )

Target phase: 0.3
Four-qubit phase resolution: 0.0625

Results ranked by frequency:
0101 -> phase 0.3125 | 3557 shots | 86.8%
0100 -> phase 0.2500 | 239 shots | 5.8%
0110 -> phase 0.3750 | 121 shots | 3.0%
0011 -> phase 0.1875 | 41 shots | 1.0%
0111 -> phase 0.4375 | 36 shots | 0.9%
0010 -> phase 0.1250 | 24 shots | 0.6%
1000 -> phase 0.5000 | 16 shots | 0.4%
0000 -> phase 0.0000 | 11 shots | 0.3%
1001 -> phase 0.5625 | 10 shots | 0.2%
0001 -> phase 0.0625 | 9 shots | 0.2%
1010 -> phase 0.6250 | 8 shots | 0.2%
1111 -> phase 0.9375 | 7 shots | 0.2%
1011 -> phase 0.6875 | 7 shots | 0.2%
1110 -> phase 0.8750 | 6 shots | 0.1%
1100 -> phase 0.7500 | 3 shots | 0.1%
1101 -> phase 0.8125 | 1 shots | 0.0%


In [13]:
# Exact-phase validation for 4 counting qubits
phi_exact = 5 / 16
m = 4
target = m

qc_exact = QuantumCircuit(m + 1, m)

# Prepare eigenstate |1>
qc_exact.x(target)

# Prepare counting register
for q in range(m):
    qc_exact.h(q)

# Encode the exact phase
for q in range(m):
    angle = 2 * np.pi * phi_exact * (2 ** q)
    qc_exact.cp(angle, q, target)

# Inverse QFT and measurement
qc_exact.append(
    QFTGate(m).inverse(),
    list(range(m))
)
qc_exact.measure(range(m), range(m))

print(qc_exact.draw())

     ┌───┐                                      ┌─────────┐┌─┐         
q_0: ┤ H ├─■────────────────────────────────────┤0        ├┤M├─────────
     ├───┤ │                                    │         │└╥┘┌─┐      
q_1: ┤ H ├─┼─────────■──────────────────────────┤1        ├─╫─┤M├──────
     ├───┤ │         │                          │  qft_dg │ ║ └╥┘┌─┐   
q_2: ┤ H ├─┼─────────┼─────────■────────────────┤2        ├─╫──╫─┤M├───
     ├───┤ │         │         │                │         │ ║  ║ └╥┘┌─┐
q_3: ┤ H ├─┼─────────┼─────────┼─────────■──────┤3        ├─╫──╫──╫─┤M├
     ├───┤ │P(5π/8)  │P(5π/4)  │P(5π/2)  │P(5π) └─────────┘ ║  ║  ║ └╥┘
q_4: ┤ X ├─■─────────■─────────■─────────■──────────────────╫──╫──╫──╫─
     └───┘                                                  ║  ║  ║  ║ 
c: 4/═══════════════════════════════════════════════════════╩══╩══╩══╩═
                                                            0  1  2  3 


In [14]:
compiled_exact = transpile(qc_exact, simulator)

result_exact = simulator.run(
    compiled_exact,
    shots=4096
).result()

counts_exact = result_exact.get_counts()

print("Exact phase:", phi_exact)
print("Measurement counts:", counts_exact)

assert counts_exact == {"0101": 4096}
print("Exact-phase validation passed!")

Exact phase: 0.3125
Measurement counts: {'0101': 4096}
Exact-phase validation passed!


In [15]:
def build_qpe(phi, m):
    """Build a QPE circuit for a known phase phi."""

    if not 0 <= phi < 1:
        raise ValueError("phi must be in [0, 1).")

    if m < 1 or not isinstance(m, int):
        raise ValueError("m must be a positive integer.")

    target = m
    qc = QuantumCircuit(m + 1, m)

    # Prepare target eigenstate |1>
    qc.x(target)

    # Create superposition in the counting register
    for q in range(m):
        qc.h(q)

    # Encode the phase using controlled powers of U
    for q in range(m):
        angle = 2 * np.pi * phi * (2 ** q)
        qc.cp(angle, q, target)

    # Apply inverse QFT
    qc.append(
        QFTGate(m).inverse(),
        list(range(m))
    )

    # Measure counting qubits
    qc.measure(range(m), range(m))

    return qc

In [16]:
phi_test = 3 / 16
m_test = 4

test_qc = build_qpe(phi_test, m_test)

print(test_qc.draw())

     ┌───┐                                      ┌─────────┐┌─┐         
q_0: ┤ H ├─■────────────────────────────────────┤0        ├┤M├─────────
     ├───┤ │                                    │         │└╥┘┌─┐      
q_1: ┤ H ├─┼─────────■──────────────────────────┤1        ├─╫─┤M├──────
     ├───┤ │         │                          │  qft_dg │ ║ └╥┘┌─┐   
q_2: ┤ H ├─┼─────────┼─────────■────────────────┤2        ├─╫──╫─┤M├───
     ├───┤ │         │         │                │         │ ║  ║ └╥┘┌─┐
q_3: ┤ H ├─┼─────────┼─────────┼─────────■──────┤3        ├─╫──╫──╫─┤M├
     ├───┤ │P(3π/8)  │P(3π/4)  │P(3π/2)  │P(3π) └─────────┘ ║  ║  ║ └╥┘
q_4: ┤ X ├─■─────────■─────────■─────────■──────────────────╫──╫──╫──╫─
     └───┘                                                  ║  ║  ║  ║ 
c: 4/═══════════════════════════════════════════════════════╩══╩══╩══╩═
                                                            0  1  2  3 


In [17]:
compiled_test = transpile(test_qc, simulator)

test_result = simulator.run(
    compiled_test,
    shots=4096
).result()

test_counts = test_result.get_counts()

print("Target phase:", phi_test)
print("Counts:", test_counts)

Target phase: 0.1875
Counts: {'0011': 4096}


In [18]:
def analyse_qpe(counts, m, target_phi):
    total_shots = sum(counts.values())

    print(f"Target phase: {target_phi:.4f}")
    print(f"Total shots: {total_shots}\n")

    for bits, shots in sorted(
        counts.items(),
        key=lambda item: item[1],
        reverse=True
    ):
        k = int(bits, 2)
        estimate = k / (2 ** m)
        error = abs(target_phi - estimate)
        frequency = shots / total_shots

        print(
            f"Bits: {bits} | "
            f"k: {k} | "
            f"Estimate: {estimate:.4f} | "
            f"Frequency: {frequency:.2%} | "
            f"Error: {error:.4f}"
        )

In [19]:
analyse_qpe(
    counts=counts4,
    m=4,
    target_phi=0.30
)

Target phase: 0.3000
Total shots: 4096

Bits: 0101 | k: 5 | Estimate: 0.3125 | Frequency: 86.84% | Error: 0.0125
Bits: 0100 | k: 4 | Estimate: 0.2500 | Frequency: 5.83% | Error: 0.0500
Bits: 0110 | k: 6 | Estimate: 0.3750 | Frequency: 2.95% | Error: 0.0750
Bits: 0011 | k: 3 | Estimate: 0.1875 | Frequency: 1.00% | Error: 0.1125
Bits: 0111 | k: 7 | Estimate: 0.4375 | Frequency: 0.88% | Error: 0.1375
Bits: 0010 | k: 2 | Estimate: 0.1250 | Frequency: 0.59% | Error: 0.1750
Bits: 1000 | k: 8 | Estimate: 0.5000 | Frequency: 0.39% | Error: 0.2000
Bits: 0000 | k: 0 | Estimate: 0.0000 | Frequency: 0.27% | Error: 0.3000
Bits: 1001 | k: 9 | Estimate: 0.5625 | Frequency: 0.24% | Error: 0.2625
Bits: 0001 | k: 1 | Estimate: 0.0625 | Frequency: 0.22% | Error: 0.2375
Bits: 1010 | k: 10 | Estimate: 0.6250 | Frequency: 0.20% | Error: 0.3250
Bits: 1111 | k: 15 | Estimate: 0.9375 | Frequency: 0.17% | Error: 0.6375
Bits: 1011 | k: 11 | Estimate: 0.6875 | Frequency: 0.17% | Error: 0.3875
Bits: 1110 | k: 14 |

In [20]:
# Use the actual four-counting-qubit experiment, not a separate scratch bit string.
most_likely_bits = max(counts4, key=counts4.get)
k = int(most_likely_bits, 2)
estimate = k / (2 ** 4)

print("Most likely measured bit string:", most_likely_bits)
print("Integer outcome k:", k)
print("Phase estimate:", estimate)

assert most_likely_bits == "0101"
assert np.isclose(estimate, 5 / 16)


Most likely measured bit string: 0101
Integer outcome k: 5
Phase estimate: 0.3125


In [21]:
# Calculate error against the target phase used by the four-qubit experiment.
target_phi = 0.30
error = abs(target_phi - estimate)

print(f"Target phase:       {target_phi:.4f}")
print(f"Estimated phase:    {estimate:.4f}")
print(f"Absolute error:     {error:.4f}")
print(f"Four-qubit grid:    {1 / 16:.4f}")

assert np.isclose(error, 0.0125)


Target phase:       0.3000
Estimated phase:    0.3125
Absolute error:     0.0125
Four-qubit grid:    0.0625


In [22]:
analyse_qpe(counts=counts4, m=4, target_phi=0.30)

Target phase: 0.3000
Total shots: 4096

Bits: 0101 | k: 5 | Estimate: 0.3125 | Frequency: 86.84% | Error: 0.0125
Bits: 0100 | k: 4 | Estimate: 0.2500 | Frequency: 5.83% | Error: 0.0500
Bits: 0110 | k: 6 | Estimate: 0.3750 | Frequency: 2.95% | Error: 0.0750
Bits: 0011 | k: 3 | Estimate: 0.1875 | Frequency: 1.00% | Error: 0.1125
Bits: 0111 | k: 7 | Estimate: 0.4375 | Frequency: 0.88% | Error: 0.1375
Bits: 0010 | k: 2 | Estimate: 0.1250 | Frequency: 0.59% | Error: 0.1750
Bits: 1000 | k: 8 | Estimate: 0.5000 | Frequency: 0.39% | Error: 0.2000
Bits: 0000 | k: 0 | Estimate: 0.0000 | Frequency: 0.27% | Error: 0.3000
Bits: 1001 | k: 9 | Estimate: 0.5625 | Frequency: 0.24% | Error: 0.2625
Bits: 0001 | k: 1 | Estimate: 0.0625 | Frequency: 0.22% | Error: 0.2375
Bits: 1010 | k: 10 | Estimate: 0.6250 | Frequency: 0.20% | Error: 0.3250
Bits: 1111 | k: 15 | Estimate: 0.9375 | Frequency: 0.17% | Error: 0.6375
Bits: 1011 | k: 11 | Estimate: 0.6875 | Frequency: 0.17% | Error: 0.3875
Bits: 1110 | k: 14 |

# Day 8 continuation — QPE validation, precision, sampling, and noise

This continuation preserves the existing QPE work and adds evidence for the remaining advanced checks.

**Workflow:** Understand → Predict → Implement → Verify → Modify → Investigate.

Run the notebook from the top in Google Colab before running these cells, so `build_qpe`, `analyse_qpe`, `simulator`, and the original imports are defined. The outputs already saved in the original notebook are historical evidence; re-running verifies reproducibility in the current environment.

**Scientific boundary:** all experiments in this notebook are simulations. They do not constitute a real-hardware result.


## 1. Precision versus counting-register size

For `m` counting qubits, QPE returns grid points spaced by

\[
\Delta\phi = \frac{1}{2^m}.
\]

This grid spacing is not a guarantee that every estimate has error less than half a grid step for every possible noisy or sampled run. For an ideal, noiseless run with a phase exactly on the grid, the expected result is deterministic. For a phase between grid points, outcomes are distributed across several estimates.

**Predict first:** for target phase `0.30`, which register size should give the finest phase grid: 2, 3, 4, or 5 counting qubits?


In [27]:
target_phi_precision = 0.30
precision_rows = []

for m_precision in [2, 3, 4, 5]:
    circuit = build_qpe(target_phi_precision, m_precision)
    compiled = transpile(circuit, simulator, optimization_level=1)
    result_precision = simulator.run(
        compiled,
        shots=4096,
        seed_simulator=20261020 + m_precision
    ).result()
    counts_precision = result_precision.get_counts()

    total = sum(counts_precision.values())
    modal_bits = max(counts_precision, key=counts_precision.get)
    modal_k = int(modal_bits, 2)
    modal_estimate = modal_k / (2 ** m_precision)
    weighted_mean = sum(
        (int(bits, 2) / (2 ** m_precision)) * shots
        for bits, shots in counts_precision.items()
    ) / total
    variance = sum(
        ((int(bits, 2) / (2 ** m_precision)) - weighted_mean) ** 2 * shots
        for bits, shots in counts_precision.items()
    ) / total

    row = {
        "m": m_precision,
        "grid_spacing": 1 / (2 ** m_precision),
        "modal_bits": modal_bits,
        "modal_estimate": modal_estimate,
        "modal_abs_error": abs(target_phi_precision - modal_estimate),
        "sample_mean": weighted_mean,
        "sample_std": np.sqrt(variance),
        "shots": total,
    }
    precision_rows.append(row)

for row in precision_rows:
    print(
        f"m={row['m']} | grid={row['grid_spacing']:.5f} | "
        f"mode={row['modal_bits']} ({row['modal_estimate']:.5f}) | "
        f"mode error={row['modal_abs_error']:.5f} | "
        f"sample mean={row['sample_mean']:.5f} | "
        f"sample std={row['sample_std']:.5f} | shots={row['shots']}"
    )

assert [row["m"] for row in precision_rows] == [2, 3, 4, 5]
assert all(row["shots"] == 4096 for row in precision_rows)
assert np.isclose(precision_rows[-1]["grid_spacing"], 1 / 32)


m=2 | grid=0.25000 | mode=01 (0.25000) | mode error=0.05000 | sample mean=0.26947 | sample std=0.10642 | shots=4096
m=3 | grid=0.12500 | mode=010 (0.25000) | mode error=0.05000 | sample mean=0.30615 | sample std=0.13477 | shots=4096
m=4 | grid=0.06250 | mode=0101 (0.31250) | mode error=0.01250 | sample mean=0.31310 | sample std=0.06009 | shots=4096
m=5 | grid=0.03125 | mode=01010 (0.31250) | mode error=0.01250 | sample mean=0.30909 | sample std=0.07120 | shots=4096


### Interpretation checkpoint

- Increasing `m` halves the phase-grid spacing each time one counting qubit is added.
- The modal estimate is the single most frequently observed bit string converted to a phase.
- The sample mean summarizes all measured outcomes; it need not equal the mode.
- The sample standard deviation describes the spread of measured phase estimates in this run.
- More counting qubits require more controlled powers of the unitary and a larger circuit. Precision therefore has a circuit-resource cost.

Record which register size gave the smallest observed modal error. Do not claim that a single finite-shot experiment proves a universal scaling law.


## 2. Exact-phase regression tests

A useful test suite checks several exactly representable phases, not just one example. With `m=4`, phases `k/16` should be recovered exactly by this ideal implementation.

**Predict first:** what bit string should phase `5/16` produce when `m=4`?


In [24]:
m_regression = 4
exact_phases = [0 / 16, 1 / 16, 3 / 16, 5 / 16, 15 / 16]
regression_results = []

for exact_phi in exact_phases:
    circuit = build_qpe(exact_phi, m_regression)
    compiled = transpile(circuit, simulator, optimization_level=1)
    counts_regression = simulator.run(
        compiled,
        shots=1024,
        seed_simulator=1234
    ).result().get_counts()

    expected_bits = format(round(exact_phi * (2 ** m_regression)), "04b")
    regression_results.append((exact_phi, expected_bits, counts_regression))
    print(
        f"phi={exact_phi:.4f} | expected={expected_bits} | "
        f"observed={counts_regression}"
    )
    assert counts_regression == {expected_bits: 1024}, (
        f"Exact-phase regression failed for phi={exact_phi}: {counts_regression}"
    )

print("All exact-phase regression tests passed.")


phi=0.0000 | expected=0000 | observed={'0000': 1024}
phi=0.0625 | expected=0001 | observed={'0001': 1024}
phi=0.1875 | expected=0011 | observed={'0011': 1024}
phi=0.3125 | expected=0101 | observed={'0101': 1024}
phi=0.9375 | expected=1111 | observed={'1111': 1024}
All exact-phase regression tests passed.


## 3. Controlled noise experiment

Now compare the same four-qubit QPE circuit under an ideal simulator and a simple illustrative noise model.

The model below applies:
- depolarizing noise after selected one-qubit gates,
- depolarizing noise after CNOT gates,
- asymmetric readout errors.

These are **chosen simulation parameters**, not calibration data from an IBM backend. They illustrate how noise can affect a phase-estimation distribution; they do not predict the performance of a particular device.

The transpiler is directed to the noisy simulator so that the noise model's supported gate basis is respected.


In [26]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.002, 1),
    ["h", "x"]
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx"]
)
readout_error = ReadoutError([
    [0.98, 0.02],
    [0.03, 0.97]
])
noise_model.add_all_qubit_readout_error(readout_error)

noisy_simulator = AerSimulator(noise_model=noise_model)
noise_target_phi = 0.30
noise_m = 4
noise_shots = 8192
noise_circuit = build_qpe(noise_target_phi, noise_m)

# Ideal baseline with the same shot count.
ideal_compiled = transpile(
    noise_circuit,
    simulator,
    optimization_level=1
)
ideal_counts_noise_comparison = simulator.run(
    ideal_compiled,
    shots=noise_shots,
    seed_simulator=20261020
).result().get_counts()

# Noisy run.
noisy_compiled = transpile(
    noise_circuit,
    noisy_simulator,
    optimization_level=1
)
noisy_counts = noisy_simulator.run(
    noisy_compiled,
    shots=noise_shots,
    seed_simulator=20261021
).result().get_counts()

def summarize_qpe_counts(counts, m, target_phi):
    total = sum(counts.values())
    mode_bits = max(counts, key=counts.get)
    mode_estimate = int(mode_bits, 2) / (2 ** m)
    mean_estimate = sum(
        int(bits, 2) / (2 ** m) * shots
        for bits, shots in counts.items()
    ) / total
    variance = sum(
        (int(bits, 2) / (2 ** m) - mean_estimate) ** 2 * shots
        for bits, shots in counts.items()
    ) / total
    return {
        "shots": total,
        "mode_bits": mode_bits,
        "mode_estimate": mode_estimate,
        "mode_frequency": counts[mode_bits] / total,
        "mode_abs_error": abs(target_phi - mode_estimate),
        "sample_mean": mean_estimate,
        "sample_std": np.sqrt(variance),
    }

ideal_summary = summarize_qpe_counts(
    ideal_counts_noise_comparison, noise_m, noise_target_phi
)
noisy_summary = summarize_qpe_counts(noisy_counts, noise_m, noise_target_phi)

print("IDEAL SIMULATION")
for key, value in ideal_summary.items():
    print(f"{key}: {value}")
print("\nNOISY SIMULATION")
for key, value in noisy_summary.items():
    print(f"{key}: {value}")

print("\nMost frequent outcomes — ideal:")
print(sorted(ideal_counts_noise_comparison.items(), key=lambda item: item[1], reverse=True)[:8])
print("\nMost frequent outcomes — noisy:")
print(sorted(noisy_counts.items(), key=lambda item: item[1], reverse=True)[:8])

assert ideal_summary["shots"] == noise_shots
assert noisy_summary["shots"] == noise_shots
assert 0 <= ideal_summary["mode_frequency"] <= 1
assert 0 <= noisy_summary["mode_frequency"] <= 1


IDEAL SIMULATION
shots: 8192
mode_bits: 0101
mode_estimate: 0.3125
mode_frequency: 0.8756103515625
mode_abs_error: 0.012500000000000011
sample_mean: 0.3126373291015625
sample_std: 0.05957015420548717

NOISY SIMULATION
shots: 8192
mode_bits: 0101
mode_estimate: 0.3125
mode_frequency: 0.65380859375
mode_abs_error: 0.012500000000000011
sample_mean: 0.3376007080078125
sample_std: 0.15620223075099973

Most frequent outcomes — ideal:
[('0101', 7173), ('0100', 453), ('0110', 200), ('0011', 101), ('0111', 56), ('0010', 49), ('1000', 31), ('0001', 28)]

Most frequent outcomes — noisy:
[('0101', 5356), ('0100', 768), ('1101', 324), ('0001', 278), ('0111', 269), ('0110', 255), ('0011', 190), ('1000', 134)]


### Noise investigation questions

Answer in your own words after running the cell:

1. Did the most likely bit string change between the ideal and noisy runs?
2. Did the fraction of shots in the most likely outcome change?
3. Did the sample spread or absolute error increase? If not, explain why noise does not have to worsen every single finite-shot metric in every run.
4. Which aspects of the model are assumptions rather than measured device facts?
5. Why should we avoid concluding that the same result will occur on real hardware?

A noisy run can occasionally give a modal estimate closer to the target by chance. Compare the distribution and repeat with recorded seeds rather than declaring that noise improves QPE from one run.



## 4. Validation summary and next-step gate

Before marking QPE complete, verify all of the following:

- [x] Notebook executed from top to bottom in Colab.
- [x] Existing exact-phase tests passed.
- [x] The scratch calculation uses the actual `counts4` experiment and target phase `0.30`.
- [x] Precision experiment completed for 2–5 counting qubits.
- [x] Exact-phase regression tests passed.
- [x] Ideal/noisy comparison ran successfully and its assumptions were documented.
- [x] I can explain the difference between grid spacing, modal estimate, sample mean, and sampling spread.
- [ ] I recorded Qiskit and Qiskit Aer versions and the simulator/noise settings used.
- [ ] I saved the executed notebook and a short Day 8 progress note.

**Status after adding these cells:** QPE extension executed; version recording and final progress documentation remain pending.

**Next roadmap task after this gate:** move into transpilation and backend optimization—compare optimization levels, circuit depth and gate counts, then study layout/routing. This follows the existing sprint plan; it is not a new roadmap.
